In [1]:
from pathlib import Path
from math import gcd
from time import perf_counter
import importlib
import inspect
import json
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

In [2]:
EVALUATION_SPLIT = "test"       
MAX_SAMPLES = 12
NUM_BACKGROUND_PILOT = 6
NUM_FRAMES = 4  # Fixed-frame reference for compute comparisons
DOWNLOAD_MISSING_MP4 = True
USE_CUDA = True
RANDOM_SEED = 42
MAMMALPS_ZIP_URL = "https://zenodo.org/records/15588220/files/mammalps_v1.zip"

# 10%-pruned FP32 model -> battery frame cap -> confidence exit.
VIDEO_MODE = "trained"
VIDEO_ADAPTIVE_THRESHOLD = 0.9
VIDEO_CHECKPOINT_OVERRIDE = "Audio/Video_checkpoints/Pruned_10/prune_0p1_full.pt"
VIDEO_CODE_DIR_OVERRIDE = None
VIDEO_METADATA_DIR_OVERRIDE = None  # Dataset metadata directory for overlap checks
BATTERY_LEVEL = 1.0  # Fraction [0, 1]; manual experiment setting, not a hardware reading


def read_battery_level():
    level = float(BATTERY_LEVEL)
    if not np.isfinite(level) or not 0 <= level <= 1:
        raise ValueError("BATTERY_LEVEL must be a fraction between 0 and 1")
    return level


def find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if any((base / name / "metadata" / "benchmark_2_audio_model_manifest.csv").is_file()
               for name in ("data", "Data")):
            return base.resolve()
    raise FileNotFoundError("Run from project root or Audio/: missing data/metadata/benchmark_2_audio_model_manifest.csv")

ROOT = find_project_root()
DATA_DIR = next(ROOT / name for name in ("data", "Data")
                if (ROOT / name / "metadata" / "benchmark_2_audio_model_manifest.csv").is_file())
VIDEO_METADATA_DIR = (Path(VIDEO_METADATA_DIR_OVERRIDE) if VIDEO_METADATA_DIR_OVERRIDE else
    next((path for path in (ROOT / "Video" / "data" / "mammalps_v1" / "benchmark_1" / "metadata",
                           DATA_DIR / "metadata") if (path / "train.csv").is_file()),
         DATA_DIR / "metadata"))
AUDIO_DIR = ROOT / "Audio"
if not AUDIO_DIR.is_dir():
    AUDIO_DIR = ROOT / "audio"
MODELS_DIR = AUDIO_DIR / "models"
RESULTS_DIR = AUDIO_DIR / "results"
MANIFEST_PATH = DATA_DIR / "metadata" / "benchmark_2_audio_model_manifest.csv"
VIDEO_CACHE_DIR = DATA_DIR / "video_cache" / "raw"
OUTPUT_DIR = AUDIO_DIR / "results" / "stage_a"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device("cuda" if USE_CUDA and torch.cuda.is_available() else "cpu")
print("Root:", ROOT)
print("Device:", DEVICE)
print("Video cache:", VIDEO_CACHE_DIR)

Root: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project
Device: cuda
Video cache: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\data\video_cache\raw


In [4]:
manifest = pd.read_csv(MANIFEST_PATH)
required = {
    "sample_id", "event_id", "file_id", "remote_video_path", "start_2s", "end_2s",
    "duration", "binary_label", "dataset_split",
}
assert required.issubset(manifest.columns), f"Manifest missing: {sorted(required - set(manifest.columns))}"
assert manifest["sample_id"].is_unique
assert np.allclose(manifest.end_2s - manifest.start_2s, 2.0, atol=0.001)
assert not manifest[list(required)].isna().any().any(), "Missing required manifest fields"
assert manifest.groupby("event_id").dataset_split.nunique().max() == 1

pool = manifest[manifest.dataset_split == EVALUATION_SPLIT].copy()
assert len(pool) > 0, f"No rows for split={EVALUATION_SPLIT!r}"

b1_train_path = VIDEO_METADATA_DIR / "train.csv"
if b1_train_path.exists():
    b1train = pd.read_csv(b1_train_path)
    assert "video_path" in b1train.columns
    b1_train_raw_ids = set(
        b1train["video_path"].astype(str).str.extract(r"(S\d+_C\d+_E\d+_V\d+)")[0].dropna()
    )
    overlap = pool.file_id.isin(b1_train_raw_ids)
    print("Test/validation audio samples from a video-training recording:", int(overlap.sum()))
    pool = pool.loc[~overlap].copy()
else:
    warnings.warn("Benchmark I data/metadata/train.csv missing: video-training overlap NOT checked")

if MAX_SAMPLES is not None:
    assert MAX_SAMPLES > 0
    bg = pool[pool.binary_label == 0].sample(n=min(NUM_BACKGROUND_PILOT, (pool.binary_label == 0).sum(), MAX_SAMPLES), random_state=RANDOM_SEED)
    animals = pool[pool.binary_label == 1].sample(n=min(MAX_SAMPLES - len(bg), (pool.binary_label == 1).sum()), random_state=RANDOM_SEED)
    selected = pd.concat([bg, animals], ignore_index=True)
else:
    selected = pool.reset_index(drop=True)

assert len(selected), "No samples selected"
print("Audio manifest total:", len(manifest))
print("Evaluation pool (after overlap filter):", len(pool))
print("Selected pilot size:", len(selected))
print(selected.binary_label.value_counts().rename({0:"background", 1:"animal"}).to_string())
display(selected[["sample_id", "file_id", "start_2s", "end_2s", "binary_label", "remote_video_path"]].head(12))

Test/validation audio samples from a video-training recording: 0
Audio manifest total: 908
Evaluation pool (after overlap filter): 207
Selected pilot size: 12
binary_label
background    6
animal        6


,sample_id,file_id,start_2s,end_2s,binary_label,remote_video_path
0,E607_1,S3_C3_E607_V0568,44.375000,46.375000,0,raw_videos/S3/C3/S3_C3_E607_V0568.mp4
1,E297_0,S2_C2_E297_V0048,29.125000,31.125000,0,raw_videos/S2/C2/S2_C2_E297_V0048.mp4
2,E549_1,S3_C3_E549_V0444,29.250000,31.250000,0,raw_videos/S3/C3/S3_C3_E549_V0444.mp4
3,E507_1,S3_C3_E507_V0292,29.250000,31.250000,0,raw_videos/S3/C3/S3_C3_E507_V0292.mp4
4,E504_0,S3_C3_E504_V0288,14.125000,16.125000,0,raw_videos/S3/C3/S3_C3_E504_V0288.mp4
5,E604_1,S3_C3_E604_V0564,29.250000,31.250000,0,raw_videos/S3/C3/S3_C3_E604_V0564.mp4
6,E153_0,S1_C2_E153_V0383,1.983333,3.983333,1,raw_videos/S1/C2/S1_C2_E153_V0383.mp4
7,E301_1,S2_C2_E301_V0052,35.983333,37.983333,1,raw_videos/S2/C2/S2_C2_E301_V0052.mp4
8,E304_0,S2_C1_E304_V0050,21.300000,23.300000,1,raw_videos/S2/C1/S2_C1_E304_V0050.mp4
9,E61_0,S1_C3_E61_V0029,8.466667,10.466667,1,raw_videos/S1/C3/S1_C3_E61_V0029.mp4


In [5]:
paths = {
    "PANNs pretrained backbone": MODELS_DIR / "pretrained" / "Cnn14_mAP=0.431.pth",
    "Final audio head": MODELS_DIR / "panns_cnn14_binary_head_2p0s.pt",
    "Audio exit 1": MODELS_DIR / "panns_exit1_block2_2p0s.pt",
    "Audio exit 2": MODELS_DIR / "panns_exit2_block4_2p0s.pt",
    "Frozen policy": RESULTS_DIR / "panns_early_exit_policy_results_2p0s.csv",
    "Frozen thresholds": RESULTS_DIR / "panns_early_exit_standalone_2p0s.csv",
    "Audio profiled FLOPs": RESULTS_DIR / "panns_audio_all_paths_flops_2p0s.csv",
}
video_code_choices = [ROOT, ROOT / "Video", ROOT / "video"]
VIDEO_CODE_DIR = (Path(VIDEO_CODE_DIR_OVERRIDE) if VIDEO_CODE_DIR_OVERRIDE else
                  next((p for p in video_code_choices if (p / "video.py").is_file()), None))
video_checkpoint_choices = [
    ROOT / "Audio" / "Video_checkpoints" / "Pruned_10" / "prune_0p1_full.pt",
    ROOT / "Video" / "checkpoints" / "prune_0p1_full.pt",
]
VIDEO_CHECKPOINT = (Path(VIDEO_CHECKPOINT_OVERRIDE) if VIDEO_CHECKPOINT_OVERRIDE else
                    next((p for p in video_checkpoint_choices if p.is_file()), video_checkpoint_choices[0]))

if not VIDEO_CHECKPOINT.is_absolute():
    VIDEO_CHECKPOINT = ROOT / VIDEO_CHECKPOINT
VIDEO_WEIGHTS_AVAILABLE = VIDEO_CHECKPOINT.is_file()
VIDEO_INFERENCE_MODE = "trained"
rows = [{"item": k, "found": path.is_file(), "path": str(path)} for k, path in paths.items()]
rows.extend([
    {"item": "video.py", "found": VIDEO_CODE_DIR is not None, "path": str(VIDEO_CODE_DIR)},
    {"item": "Pruned FP32 video checkpoint", "found": VIDEO_WEIGHTS_AVAILABLE, "path": str(VIDEO_CHECKPOINT)},

])
preflight = pd.DataFrame(rows)
display(preflight)
missing = preflight.loc[~preflight.found, ["item", "path"]]
if len(missing):
    raise FileNotFoundError("Missing pipeline inputs:\n" + missing.to_string(index=False))
if not 0 <= VIDEO_ADAPTIVE_THRESHOLD <= 1:
    raise ValueError("VIDEO_ADAPTIVE_THRESHOLD must be in [0, 1]")
read_battery_level()


,item,found,path
0,PANNs pretrained backbone,True,C:\Users\benja\Documents\CS_Master\Big Data at...
1,Final audio head,True,C:\Users\benja\Documents\CS_Master\Big Data at...
2,Audio exit 1,True,C:\Users\benja\Documents\CS_Master\Big Data at...
3,Audio exit 2,True,C:\Users\benja\Documents\CS_Master\Big Data at...
4,Frozen policy,True,C:\Users\benja\Documents\CS_Master\Big Data at...
5,Frozen thresholds,True,C:\Users\benja\Documents\CS_Master\Big Data at...
6,Audio profiled FLOPs,True,C:\Users\benja\Documents\CS_Master\Big Data at...
7,video.py,True,C:\Users\benja\Documents\CS_Master\Big Data at...
8,Pruned FP32 video checkpoint,True,C:\Users\benja\Documents\CS_Master\Big Data at...


1.0

In [6]:
from urllib.request import urlretrieve
PANNS_LABELS = Path.home() / "panns_data" / "class_labels_indices.csv"
if not PANNS_LABELS.exists():
    PANNS_LABELS.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve("https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv", PANNS_LABELS)

from panns_inference import AudioTagging

POLICY = pd.read_csv(paths["Frozen policy"])
row_policy = POLICY[(POLICY.operating_point == "balanced") & (POLICY.split == "validation")]
assert len(row_policy) == 1, "Expected exactly one balanced validation policy"
MARGIN1 = float(row_policy.iloc[0].margin1)
MARGIN2 = float(row_policy.iloc[0].margin2)

standalone = pd.read_csv(paths["Frozen thresholds"])
val_thresholds = standalone[standalone.split == "validation"]
def threshold_for(label):
    subset = val_thresholds[val_thresholds["exit"].str.lower().str.contains(label)]
    assert len(subset) == 1, f"Cannot find exactly one validation threshold for {label!r}"
    return float(subset.iloc[0].threshold)
T1, T2, T_FINAL = threshold_for("block 2"), threshold_for("block 4"), threshold_for("final")
print("Balanced policy margins:", MARGIN1, MARGIN2)
print("Frozen thresholds:", T1, T2, T_FINAL)

panns = AudioTagging(checkpoint_path=str(paths["PANNs pretrained backbone"]), device=DEVICE.type)
BACKBONE = panns.model.module if isinstance(panns.model, nn.DataParallel) else panns.model
BACKBONE.eval().requires_grad_(False)

class BinaryHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.30):
        super().__init__()
        self.network = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim),
                                     nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        return self.network(x).squeeze(-1)

def load_head(path, state_field):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    head = BinaryHead(ck["input_dim"], ck["hidden_dim"], ck["dropout"]).to(DEVICE).eval()
    head.load_state_dict(ck[state_field], strict=True)
    head.requires_grad_(False)
    return head

EXIT1_HEAD = load_head(paths["Audio exit 1"], "state_dict")
EXIT2_HEAD = load_head(paths["Audio exit 2"], "state_dict")
FINAL_HEAD = load_head(paths["Final audio head"], "head_state_dict")

def pooled(x):
    x = x.mean(dim=3)
    return x.max(dim=2).values + x.mean(dim=2)

def logit_margin(p, threshold):
    eps = 1e-6
    p, threshold = np.clip([p, threshold], eps, 1-eps)
    return float(np.log(p / (1-p)) - np.log(threshold / (1-threshold)))

@torch.inference_mode()
def balanced_audio_predict(waveform_32k):
    wav = torch.from_numpy(waveform_32k[None,:]).float().to(DEVICE)
    m = BACKBONE
    x = m.logmel_extractor(m.spectrogram_extractor(wav))
    x = m.bn0(x.transpose(1,3)).transpose(1,3)
    x = m.conv_block1(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block2(x, pool_size=(2,2), pool_type="avg")
    p1 = float(torch.sigmoid(EXIT1_HEAD(pooled(x))).item())
    d1 = logit_margin(p1, T1)
    if abs(d1) >= MARGIN1:
        return {"audio_prediction": int(d1 >= MARGIN1), "audio_probability": p1, "exit_stage": 1}
    x = m.conv_block3(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block4(x, pool_size=(2,2), pool_type="avg")
    p2 = float(torch.sigmoid(EXIT2_HEAD(pooled(x))).item())
    d2 = logit_margin(p2, T2)
    if abs(d2) >= MARGIN2:
        return {"audio_prediction": int(d2 >= MARGIN2), "audio_probability": p2, "exit_stage": 2}
    x = m.conv_block5(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block6(x, pool_size=(1,1), pool_type="avg")
    x = x.mean(dim=3)
    emb = F.relu(m.fc1(x.max(dim=2).values + x.mean(dim=2)))
    prob = float(torch.sigmoid(FINAL_HEAD(emb)).item())
    return {"audio_prediction": int(prob >= T_FINAL), "audio_probability": prob, "exit_stage": 3}

Balanced policy margins: 2.0 0.25
Frozen thresholds: 0.2324143648147583 0.2000872790813446 0.1661750376224517
Checkpoint path: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\models\pretrained\Cnn14_mAP=0.431.pth


C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU number: 1


In [7]:
AUDIO_SAMPLE_RATE = 32000
AUDIO_SECONDS = 2.0

def get_audio_path(row):
    return DATA_DIR / "audio_cache" / "full" / f"{row['file_id']}.flac"

def read_audio_window(row):
    audio_path = get_audio_path(row)
    if not audio_path.is_file():
        raise FileNotFoundError(f"Cached soundtrack missing: {audio_path}")
    start, end = float(row["start_2s"]), float(row["end_2s"])
    assert abs((end - start) - AUDIO_SECONDS) <= 0.001
    with sf.SoundFile(str(audio_path)) as sfh:
        source_sr = int(sfh.samplerate)
        sfh.seek(round(start * source_sr))
        x = sfh.read(frames=round(AUDIO_SECONDS * source_sr),
                     dtype="float32", always_2d=False)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if source_sr != AUDIO_SAMPLE_RATE:
        d = gcd(source_sr, AUDIO_SAMPLE_RATE)
        x = resample_poly(x, up=AUDIO_SAMPLE_RATE // d, down=source_sr // d)
    expected = round(AUDIO_SECONDS * AUDIO_SAMPLE_RATE)
    if len(x) < expected:
        x = np.pad(x, (0, expected-len(x)))
    x = x[:expected]
    return np.clip(x, -1, 1).astype(np.float32)

print("Cached soundtracks present in selection:", sum(get_audio_path(r).is_file() for _,r in selected.iterrows()), "/", len(selected))

Cached soundtracks present in selection: 12 / 12


In [8]:
sys.path.insert(0, str(VIDEO_CODE_DIR.resolve()))
video_module = importlib.import_module("video")
from adaptive_video import select_policy_frames
from torchvision.models import MobileNet_V3_Large_Weights
from utils import count_flops

VIDEO_DEVICE = torch.device("cpu")
# Trusted project checkpoint: full model is necessary for its pruned architecture.
video_ck = torch.load(str(VIDEO_CHECKPOINT), map_location="cpu", weights_only=False)
if "model" not in video_ck or video_ck.get("target") != "activity":
    raise ValueError("Expected a full pruned activity checkpoint (*_full.pt)")
video_info = {key: value for key, value in video_ck.items() if key != "model"}
class_to_idx = video_info["class_to_idx"]
if not class_to_idx or sorted(class_to_idx.values()) != list(range(len(class_to_idx))):
    raise ValueError("Checkpoint must store a contiguous class_to_idx mapping")
idx_to_class = {int(v): k for k, v in class_to_idx.items()}
VIDEO_TRANSFORM = MobileNet_V3_Large_Weights.DEFAULT.transforms()
if video_info.get("preprocessing") != str(MobileNet_V3_Large_Weights.DEFAULT):
    raise ValueError("Checkpoint preprocessing does not match the video transform")
if video_info.get("aggregation") != "mean_logits":
    raise ValueError("Checkpoint must use mean_logits aggregation")
VIDEO_MODEL = video_ck["model"].to(VIDEO_DEVICE).eval()
VIDEO_MODEL.device = VIDEO_DEVICE
VIDEO_GFLOPS_PER_FRAME = count_flops(VIDEO_MODEL.model, (1, 3, 224, 224), VIDEO_DEVICE) / 1e9
VIDEO_GFLOPS = VIDEO_GFLOPS_PER_FRAME * NUM_FRAMES
ADAPTIVE_SUPPORTED = {"adaptive_threshold", "return_frames_used"}.issubset(
    inspect.signature(VIDEO_MODEL.forward).parameters)
if not ADAPTIVE_SUPPORTED:
    raise ValueError("Update Video/video.py to support confidence-based frame exits")
print("Pruned FP32 model loaded:", VIDEO_CHECKPOINT)
print("Pruning ratio:", video_info.get("pruning_ratio"))
print("Video device:", VIDEO_DEVICE)
print("Battery frame budget:", select_policy_frames(read_battery_level()))
print("Confidence threshold:", VIDEO_ADAPTIVE_THRESHOLD)


C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Unsupported operator aten::hardswish_ encountered 21 time(s)
Unsupported operator aten::add_ encountered 10 time(s)
Unsupported operator aten::hardsigmoid encountered 8 time(s)
Unsupported operator aten::mul encountered 8 time(s)
Unsupported operator aten::dropout_ encountered 1 time(s)


Pruned FP32 model loaded: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\Video_checkpoints\Pruned_10\prune_0p1_full.pt
Pruning ratio: 0.1
Video device: cpu
Battery frame budget: 8
Confidence threshold: 0.9


In [9]:
import av
from contextlib import contextmanager

@contextmanager
def remote_video_cache():
    opened = None
    try:
        def retrieve(row):
            nonlocal opened
            remote = str(row["remote_video_path"]).replace("\\", "/")
            path = VIDEO_CACHE_DIR / f"{row['file_id']}.mp4"
            if path.is_file() and path.stat().st_size > 0:
                return path
            if not DOWNLOAD_MISSING_MP4:
                raise FileNotFoundError(f"Raw video not cached: {path}")
            if opened is None:
                from remotezip import RemoteZip
                opened = RemoteZip(MAMMALPS_ZIP_URL)
            partial = path.with_suffix(".download")
            try:
                with opened.open(remote) as src, partial.open("wb") as dst:
                    shutil.copyfileobj(src, dst, length=1024*1024)
                if partial.stat().st_size == 0:
                    raise ValueError(f"Downloaded an empty video: {remote}")
                partial.replace(path)
            except Exception:
                partial.unlink(missing_ok=True)
                raise
            return path
        yield retrieve
    finally:
        if opened is not None:
            opened.close()

def synchronized_frames(video_path, start_s, end_s, num_frames):
    # Nested ordering must match the video model's adaptive progression.
    if VIDEO_ADAPTIVE_THRESHOLD is not None:
        positions = video_module.nested_frame_positions(num_frames)
        # Keep requested times inside the source interval (rather than full raw MP4).
        target_times = float(start_s) + (float(end_s) - float(start_s)) * np.asarray(positions)
    else:
        target_times = np.linspace(float(start_s), float(end_s), num_frames+2)[1:-1]
    best_delta = [float("inf")] * num_frames
    best_frame = [None] * num_frames
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for frame in container.decode(video=0):
            sec = frame.time
            if sec is None and frame.pts is not None:
                sec = float(frame.pts * stream.time_base)
            if sec is None:
                continue
            for j, tgt in enumerate(target_times):
                delta = abs(float(sec) - float(tgt))
                if delta < best_delta[j]:
                    best_delta[j] = delta
                    best_frame[j] = frame.to_ndarray(format="rgb24")
            if sec > float(end_s) + 0.5 and all(f is not None for f in best_frame):
                break
    if any(f is None for f in best_frame):
        raise RuntimeError(f"Video has no timestamped frames: {video_path}")
    if max(best_delta) > 0.5:
        raise RuntimeError(f"Frames too far from requested times: {max(best_delta):.3f} s")
    return torch.stack([VIDEO_TRANSFORM(torch.from_numpy(rgb).permute(2, 0, 1))
                        for rgb in best_frame]).unsqueeze(0)

@torch.inference_mode()
def execute_video_model(frames):
    inp = frames.to(VIDEO_DEVICE)
    logits, counts = VIDEO_MODEL(
        inp, adaptive_threshold=VIDEO_ADAPTIVE_THRESHOLD, return_frames_used=True)
    frames_used = int(counts[0].item())
    probs = logits.softmax(dim=-1)[0].detach().cpu().numpy()
    index = int(probs.argmax())
    return {"activity_prediction": idx_to_class[index],
            "activity_confidence": float(probs[index]),
            "video_frames_used": frames_used}


In [10]:
audio_flops = pd.read_csv(paths["Audio profiled FLOPs"])
flops_by_path = dict(zip(audio_flops.path, audio_flops.gflops_est))
needed = ("adaptive_exit1", "adaptive_exit2", "adaptive_final")
assert all(k in flops_by_path for k in needed)
AUDIO_GFLOPS = {1: float(flops_by_path["adaptive_exit1"]),
                2: float(flops_by_path["adaptive_exit2"]),
                3: float(flops_by_path["adaptive_final"])}

print("Audio GFLOPs by actual exit:", AUDIO_GFLOPS)
print(f"Video single-frame equivalent GOPs (FP32 architecture estimate): {VIDEO_GFLOPS_PER_FRAME:.4f}")
print(f"Video {NUM_FRAMES}-frame static equivalent GOPs (FP32 architecture estimate): {VIDEO_GFLOPS:.4f}")
assert VIDEO_GFLOPS > 0


Audio GFLOPs by actual exit: {1: 2.821031552, 2: 5.657237376, 3: 8.385780096}
Video single-frame equivalent GOPs (FP32 architecture estimate): 0.1831
Video 4-frame static equivalent GOPs (FP32 architecture estimate): 0.7323


In [11]:
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# Warm up the FP32 CPU video model, independently of audio's CUDA device.
with torch.inference_mode():
    _ = VIDEO_MODEL.model(torch.zeros((1, 3, 224, 224), device=VIDEO_DEVICE))

records = []
with remote_video_cache() as ensure_video:
    for k, (_, row) in enumerate(selected.iterrows(), start=1):
        battery_level = read_battery_level()
        max_video_frames = select_policy_frames(battery_level)
        out = {"battery_level": battery_level, "video_max_frames": max_video_frames,
               "video_precision": "FP32", "video_device": str(VIDEO_DEVICE),
               "video_early_exit": False, "sample_id": row["sample_id"], "event_id": row["event_id"],
               "file_id": row["file_id"], "split": row["dataset_split"],
               "binary_label": int(row["binary_label"]),
               "start_2s": float(row["start_2s"]), "end_2s": float(row["end_2s"]),
               "video_inference_mode": VIDEO_INFERENCE_MODE,
               "video_status": "skipped", "activity_prediction": None,
               "activity_confidence": np.nan, "video_frames_used": 0,
               "video_download_ms": np.nan, "video_decode_ms": 0.0, "video_model_ms": 0.0}
        try:
            t0 = perf_counter()
            waveform = read_audio_window(row)
            out["audio_preprocess_ms"] = (perf_counter() - t0)*1000
            sync_device()
            t0 = perf_counter()
            decision = balanced_audio_predict(waveform)
            sync_device()
            out["audio_model_ms"] = (perf_counter() - t0)*1000
            out.update(decision)
            out["audio_gflops_est"] = AUDIO_GFLOPS[decision["exit_stage"]]
            out["video_activated"] = bool(decision["audio_prediction"])
            out["video_gflops_est"] = 0.0
            if out["video_activated"]:
                t0 = perf_counter()
                video_path = ensure_video(row)
                out["video_download_ms"] = (perf_counter() - t0)*1000
                t0 = perf_counter()
                frames = synchronized_frames(video_path, row["start_2s"], row["end_2s"], max_video_frames)
                out["video_decode_ms"] = (perf_counter() - t0)*1000
                sync_device()
                t0 = perf_counter()
                out.update(execute_video_model(frames))
                out["video_early_exit"] = out["video_frames_used"] < max_video_frames
                sync_device()
                out["video_model_ms"] = (perf_counter() - t0)*1000
                out["video_gflops_est"] = VIDEO_GFLOPS_PER_FRAME * out["video_frames_used"]
                out["video_status"] = "executed"
            out["run_status"] = "ok"
        except Exception as exc:
            out["run_status"] = "error"
            out["error"] = f"{type(exc).__name__}: {exc}"
        out["total_model_gflops_est"] = out.get("audio_gflops_est", np.nan) + out.get("video_gflops_est", np.nan)
        out["local_total_latency_ms"] = (
            out.get("audio_preprocess_ms", 0) + out.get("audio_model_ms", 0) +
            out.get("video_decode_ms", 0) + out.get("video_model_ms", 0)
        )
        records.append(out)
        print(f"{k}/{len(selected)} {row['sample_id']} | "
              f"audio={out.get('audio_prediction','ERROR')} exit={out.get('exit_stage','?')} "
              f"vision={out['video_status']} frames={out['video_frames_used']} "
              f"{out.get('activity_prediction') or ''} {out.get('error','')}")

results = pd.DataFrame(records)
OUTPUT_PATH = OUTPUT_DIR / f"stage_a_predictions_{EVALUATION_SPLIT}_{len(selected)}_{VIDEO_CHECKPOINT.stem}_{VIDEO_INFERENCE_MODE}_fp32_battery.csv"
results.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
display(results.head())


1/12 E607_1 | audio=0 exit=2 vision=skipped frames=0  
2/12 E297_0 | audio=1 exit=2 vision=executed frames=8 unknown 
3/12 E549_1 | audio=0 exit=2 vision=skipped frames=0  
4/12 E507_1 | audio=0 exit=2 vision=skipped frames=0  
5/12 E504_0 | audio=0 exit=2 vision=skipped frames=0  
6/12 E604_1 | audio=0 exit=2 vision=skipped frames=0  
7/12 E153_0 | audio=1 exit=1 vision=executed frames=8 vigilance 
8/12 E301_1 | audio=1 exit=1 vision=executed frames=8 vigilance 
9/12 E304_0 | audio=1 exit=2 vision=executed frames=8 unknown 
10/12 E61_0 | audio=1 exit=1 vision=executed frames=8 unknown 
11/12 E131_1 | audio=1 exit=1 vision=executed frames=8 foraging 
12/12 E293_1 | audio=1 exit=1 vision=executed frames=8 unknown 
Saved: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\results\stage_a\stage_a_predictions_test_12_prune_0p1_full_trained_fp32_battery.csv


,battery_level,video_max_frames,video_precision,video_device,video_early_exit,sample_id,event_id,file_id,split,binary_label,...,audio_model_ms,audio_prediction,audio_probability,exit_stage,audio_gflops_est,video_activated,video_gflops_est,run_status,total_model_gflops_est,local_total_latency_ms
0,1.0,8,FP32,cpu,False,E607_1,E607,S3_C3_E607_V0568,test,0,...,448.7801,0,0.002867,2,5.657237,False,0.000000,ok,5.657237,454.2675
1,1.0,8,FP32,cpu,False,E297_0,E297,S2_C2_E297_V0048,test,0,...,10.5995,1,0.700919,2,5.657237,True,1.464547,ok,7.121784,12637.1982
2,1.0,8,FP32,cpu,False,E549_1,E549,S3_C3_E549_V0444,test,0,...,10.9155,0,0.004461,2,5.657237,False,0.000000,ok,5.657237,16.4517
3,1.0,8,FP32,cpu,False,E507_1,E507,S3_C3_E507_V0292,test,0,...,10.1939,0,0.004042,2,5.657237,False,0.000000,ok,5.657237,13.8291
4,1.0,8,FP32,cpu,False,E504_0,E504,S3_C3_E504_V0288,test,0,...,12.6273,0,0.097211,2,5.657237,False,0.000000,ok,5.657237,16.3785


In [12]:
print("Processing statuses:")
print(results.run_status.value_counts(dropna=False).to_string())
if "error" in results.columns:
    display(results.loc[results.run_status != "ok", ["sample_id", "error"]])

ok = results.loc[results.run_status == "ok"].copy()
assert len(ok), "Nothing completed. Inspect logged errors above."
print(f"Completed: {len(ok)}/{len(results)}")
print("Video mode:", VIDEO_INFERENCE_MODE, "— activity predictions meaningful?", VIDEO_INFERENCE_MODE == "trained")
print(f"Video activation rate: {ok.video_activated.mean():.1%}")
print(f"Executed video inference: {(ok.video_status == 'executed').sum()}")
print("Audio exits:", ok.exit_stage.value_counts().sort_index().to_dict())
print(f"Mean combined equivalent GOPs (architecture estimate, THOP ×2): {ok.total_model_gflops_est.mean():.3f}")
print(f"Video-only fixed-{NUM_FRAMES}-frame reference equivalent GOPs: {VIDEO_GFLOPS:.3f}")
print(f"Mean local latency incl. audio preprocessing + video decode: {ok.local_total_latency_ms.mean():.1f} ms")
if (ok.binary_label == 1).any():
    print(f"Animal recall of audio gate: {ok.loc[ok.binary_label == 1,'audio_prediction'].mean():.1%}")
if (ok.binary_label == 0).any():
    print(f"Background rejection of audio gate: {1-ok.loc[ok.binary_label == 0,'audio_prediction'].mean():.1%}")
if (ok.video_status == "executed").any():
    print(f"Average frames when vision executes: {ok.loc[ok.video_status == 'executed','video_frames_used'].mean():.2f}")
assert (ok.video_frames_used <= ok.video_max_frames).all()
assert not ((ok.audio_prediction == 0) & (ok.video_status == "executed")).any()
assert not ((ok.audio_prediction == 1) & (ok.video_status != "executed")).any()
if VIDEO_INFERENCE_MODE == "profile_only":
    assert ok.activity_prediction.isna().all(), "Profiling-only must not output activity predictions"
    print("VIDEO OUTPUTS ARE FROM UNTRAINED WEIGHTS: no activity-quality claims")
if MAX_SAMPLES is not None:
    print("PILOT ONLY: increase MAX_SAMPLES for a representative workload")
print("No activity accuracy is claimed on unmatched raw frames")


Processing statuses:
run_status
ok    12
Completed: 12/12
Video mode: trained — activity predictions meaningful? True
Video activation rate: 58.3%
Executed video inference: 7
Audio exits: {1: 5, 2: 7}
Mean combined equivalent GOPs (architecture estimate, THOP ×2): 5.330
Video-only fixed-4-frame reference equivalent GOPs: 0.732
Mean local latency incl. audio preprocessing + video decode: 3924.3 ms
Animal recall of audio gate: 100.0%
Background rejection of audio gate: 83.3%
Average frames when vision executes: 8.00
PILOT ONLY: increase MAX_SAMPLES for a representative workload
No activity accuracy is claimed on unmatched raw frames


In [13]:
run_info = {
    "objective": "Stage A conditional model-execution measurement; activity accuracy not evaluated",
    "split": EVALUATION_SPLIT, "num_selected": len(selected),
    "num_completed": int((results.run_status == "ok").sum()),
    "audio_policy": "balanced", "audio_margin1": MARGIN1, "audio_margin2": MARGIN2,
    "audio_thresholds": {"exit1": T1, "exit2": T2, "final": T_FINAL},
    "video_architecture": "MobileNetV3 Large, frame-logit average",
    "video_pruning_ratio": video_info.get("pruning_ratio", 0.0),
    "video_precision": "FP32", "video_device": str(VIDEO_DEVICE),
    "video_export_metadata": video_info,
    "battery_source": "Manual BATTERY_LEVEL setting, read per sample",
    "battery_frame_policy": {">0.75": 8, ">0.50": 4, ">0.25": 2, "otherwise": 1},
    "video_inference_mode": VIDEO_INFERENCE_MODE,
    "video_checkpoint": str(VIDEO_CHECKPOINT) if VIDEO_WEIGHTS_AVAILABLE and VIDEO_INFERENCE_MODE == "trained" else None,
    "activity_predictions_valid": VIDEO_INFERENCE_MODE == "trained",
    "video_adaptive_threshold": VIDEO_ADAPTIVE_THRESHOLD,
    "video_reference_num_frames": NUM_FRAMES,
    "decoding": "All budgeted candidate frames decoded before confidence exits; exits save model work only",
    "frame_sampling": "nested positions inside 2-second window" if VIDEO_ADAPTIVE_THRESHOLD is not None else "interior evenly spaced within 2-second window",
    "flops": "Equivalent GOPs from FP32 architecture THOP MACs*2; sum audio exit and executed video frames. Not measured energy; excludes decoding/preprocessing",
    "audio_source": str(MANIFEST_PATH.relative_to(ROOT)),
    "torch": str(torch.__version__), "device": str(DEVICE),
    "download_missing_mp4": bool(DOWNLOAD_MISSING_MP4),
    "video_training_overlap_checked": bool(b1_train_path.exists()),
    "prediction_csv": str(OUTPUT_PATH.relative_to(ROOT)),
}
run_json = OUTPUT_PATH.with_suffix(".json")
run_json.write_text(json.dumps(run_info, indent=2), encoding="utf-8")
print("Saved run settings:", run_json)


Saved run settings: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\results\stage_a\stage_a_predictions_test_12_prune_0p1_full_trained_fp32_battery.json
